In [3]:
pip install fastapi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: C:\Users\animu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
pip install paddlepaddle-gpu==2.6.2

  Using cached paddlepaddle_gpu-2.6.2-cp311-cp311-win_amd64.whl.metadata (8.8 kB)
Using cached paddlepaddle_gpu-2.6.2-cp311-cp311-win_amd64.whl (477.4 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: C:\Users\animu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
# ocr_service.py

import asyncio
from typing import List

import cv2
import numpy as np
from fastapi import FastAPI, File, UploadFile
from paddleocr import PaddleOCR

app = FastAPI()

# ✅ Load model ONCE at startup
ocr = PaddleOCR(use_angle_cls=False, device="gpu")  # use device="cpu" if no GPU


def run_ocr(img: np.ndarray) -> str:
    """Runs OCR on a single image and returns concatenated text."""
    result = ocr.predict(img)

    if not result:
        return ""

    # PaddleOCR 3.x returns a list of result objects (one per image)
    res = result[0]

    # res behaves like a dict with keys: rec_texts, rec_scores, dt_polys, etc.
    texts = res.get("rec_texts", [])
    return " ".join(texts)


@app.post("/ocr/batch")
async def ocr_batch(files: List[UploadFile] = File(...)):
    loop = asyncio.get_event_loop()
    results = []

    for file in files:
        contents = await file.read()

        if not contents:
            results.append({
                "filename": file.filename,
                "text": "",
                "error": "empty file"
            })
            continue

        nparr = np.frombuffer(contents, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        if img is None:
            results.append({
                "filename": file.filename,
                "text": "",
                "error": "invalid or unsupported image"
            })
            continue

        try:
            # Offload blocking OCR call to a thread so it doesn't block the event loop
            extracted_text = await loop.run_in_executor(None, run_ocr, img)
        except Exception as e:
            results.append({
                "filename": file.filename,
                "text": "",
                "error": str(e)
            })
            continue

        results.append({
            "filename": file.filename,
            "text": extracted_text
        })

    return {"results": results}

C:\Users\animu\AppData\Local\Temp\ipykernel_14400\1443935645.py:14: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=False, device="gpu")  # use device="cpu" if no GPU
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\animu\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Error: Can not import paddle core while this file exists: C:\Users\animu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\paddle\base\libpaddle.pyd


ImportError: generic_type: type "ProgramDescTracer" is already registered!

In [3]:
pip list | findstr paddle

paddleocr                     3.7.0
paddlex                       3.7.2
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ocr_agent.py

import os
import requests

OCR_API_URL = "http://localhost:8000/ocr/batch"
BATCH_SIZE = 16


def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


def call_ocr_api(image_paths):
    files = []

    for path in image_paths:
        files.append(
            ("files", (os.path.basename(path), open(path, "rb"), "image/jpeg"))
        )

    response = requests.post(OCR_API_URL, files=files)
    return response.json()


def run_agent(image_folder):
    all_images = [
        os.path.join(image_folder, f)
        for f in os.listdir(image_folder)
        if f.lower().endswith((".jpg", ".png", ".jpeg"))
    ]

    all_results = []

    for batch in chunk_list(all_images, BATCH_SIZE):
        result = call_ocr_api(batch)
        all_results.extend(result["results"])

    return all_results


if __name__ == "__main__":
    results = run_agent("./data/Ảnh hoàn/Ảnh hoàn/")

    for r in results:
        print(r["filename"], "=>", r["text"])


ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /ocr/batch (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001990E601750>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))